In [13]:
# =========================
# 1. IMPORT LIBRARY
# =========================
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import kagglehub
tf.config.optimizer.set_jit(True)  # speed boost

In [14]:
# =========================
# 2. DATA LOADING
# =========================
IMG_SIZE = 160
BATCH_SIZE = 16

path = kagglehub.dataset_download("tushar5harma/plant-village-dataset-updated")
data_dir = path # <-- ganti sesuai path kamu

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# 🔥 BATASI DATA BIAR CEPAT
train_ds = train_ds.take(300)
val_ds = val_ds.take(100)

Using Colab cache for faster access to the 'plant-village-dataset-updated' dataset.
Found 67118 files belonging to 9 classes.
Using 53695 files for training.
Found 67118 files belonging to 9 classes.
Using 13423 files for validation.


In [15]:
# =========================
# 3. DATA AUGMENTATION
# =========================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [16]:
# =========================
# 4. CNN BACKBONE
# =========================
base_cnn = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    weights="imagenet"
)

base_cnn.trainable = False

In [17]:
# =========================
# 5. PATCH EMBEDDING
# =========================
class PatchEmbedding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.projection = layers.Dense(embed_dim)
        self.pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=embed_dim)

    def call(self, patch):
        positions = tf.range(start=0, limit=tf.shape(patch)[1], delta=1)
        return self.projection(patch) + self.pos_embedding(positions)


In [18]:
# =========================
# 6. TRANSFORMER BLOCK
# =========================
def transformer_block(x, embed_dim, num_heads, ff_dim):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(ff_dim, activation="relu")(x)
    ffn = layers.Dense(embed_dim)(ffn)
    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    return x

In [19]:
# =========================
# 7. BUILD MODEL
# =========================
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)
x = base_cnn(x, training=False)

# flatten jadi patch
x = layers.Reshape((-1, x.shape[-1]))(x)

num_patches = x.shape[1]
embed_dim = 64   # 🔥 lebih ringan

x = PatchEmbedding(num_patches, embed_dim)(x)

# 🔥 transformer diperkecil
for _ in range(2):
    x = transformer_block(x, embed_dim, num_heads=4, ff_dim=128)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)


In [20]:
# =========================
# 8. COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 160, 160,  │          0 │ input_layer_4[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 5, 5,      │  4,049,571 │ sequential_1[0][… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 25, 1280)  │          0 │ efficientnetb0[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embedding_1   │ (None, 25, 64)    │     83,584 │ reshape_1[0][0]   │
│ (PatchEmbedding)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 25, 64)    │     66,368 │ patch_embedding_… │
│ (MultiHeadAttentio… │                   │            │ patch_embedding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 25, 64)    │          0 │ patch_embedding_… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 25, 64)    │        128 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 25, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 25, 64)    │      8,256 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 25, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 25, 64)    │        128 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 25, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 25, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 25, 64)    │        128 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 25, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 25, 64)    │      8,256 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 25, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,300,140 (16.40 MB)

 Trainable params: 250,569 (978.79 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [21]:
# =========================
# 9. TRAINING
# =========================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
300/300 ━━━━━━━━━━━━━━━━━━━━ 300s 933ms/step - accuracy: 0.6667 - loss: 1.0399 - val_accuracy: 0.8669 - val_loss: 0.4849
Epoch 2/5
300/300 ━━━━━━━━━━━━━━━━━━━━ 264s 879ms/step - accuracy: 0.8792 - loss: 0.4104 - val_accuracy: 0.9100 - val_loss: 0.3043
Epoch 3/5
300/300 ━━━━━━━━━━━━━━━━━━━━ 264s 879ms/step - accuracy: 0.9198 - loss: 0.2699 - val_accuracy: 0.9225 - val_loss: 0.2517
Epoch 4/5
300/300 ━━━━━━━━━━━━━━━━━━━━ 292s 974ms/step - accuracy: 0.9371 - loss: 0.2120 - val_accuracy: 0.9256 - val_loss: 0.2413
Epoch 5/5
300/300 ━━━━━━━━━━━━━━━━━━━━ 267s 891ms/step - accuracy: 0.9460 - loss: 0.1721 - val_accuracy: 0.9331 - val_loss: 0.1961


In [22]:
# =========================
# 10. FINE-TUNING
# =========================
base_cnn.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 817s 3s/step - accuracy: 0.6460 - loss: 1.0851 - val_accuracy: 0.8012 - val_loss: 0.6417
Epoch 2/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 788s 3s/step - accuracy: 0.8058 - loss: 0.5990 - val_accuracy: 0.8331 - val_loss: 0.5045
Epoch 3/3
300/300 ━━━━━━━━━━━━━━━━━━━━ 757s 3s/step - accuracy: 0.8552 - loss: 0.4508 - val_accuracy: 0.8675 - val_loss: 0.4076


In [23]:
# =========================
# 11. EVALUATION
# =========================
loss, acc = model.evaluate(val_ds)
print("Final Validation Accuracy:", acc)

100/100 ━━━━━━━━━━━━━━━━━━━━ 52s 516ms/step - accuracy: 0.8712 - loss: 0.4046
Final Validation Accuracy: 0.8712499737739563


In [24]:
# =========================
# 12. SAVE MODEL
# =========================
model.save("hybrid_cnn_vit_fast.h5")